# Annual-report RAG on Colab: two ingestion pipelines

Choose a pipeline in cell 1, then **Runtime → Run all**. All the logic lives in
[scripts/rag/colab_pipeline.py](../scripts/rag/colab_pipeline.py); this notebook only sets it up.

| | `colpali_qdrant` | `vlm_chroma` |
|---|---|---|
| Table, chart and image pages | ColPali page vectors in `annual_reports_visual` | a VLM (Qwen3-VL) writes them as Markdown text: tables as tables, charts as labelled values |
| Unreadable text pages (broken text layer, ~25%) | OCR text into `annual_reports_text` | the VLM writes them as text too (this pipeline has no OCR) |
| All the text | its own collection `annual_reports_text`: text embeddings + number-aware BM25 | text layer, chunked; **every chunk is a text embedding** in Chroma |
| **RAM** | text vectors; ColPali **mean-pooled** rows + cols (38 × 128 each) with HNSW | Chroma's HNSW, loaded when queried |
| **Disk** | ColPali **full** multivectors (1030 × 128, float16, mmap, rerank only); payloads | Chroma (SQLite + HNSW files) |
| DB | Qdrant server on this VM | Chroma |
| GPU | ColPali: T4 is fine (L4 preferred) | Qwen3-VL-4B 4-bit: T4 (2B faster); 8B: L4/A100 |

**Where data lives:**
- The live index sits on this VM's fast disk (`/content/rag`).
- Drive (`MyDrive/capstone_rag`) holds the code zip, the PDFs (by sector), the VLM transcriptions, and a copy of the index: a Qdrant snapshot or the Chroma folder after every sector.
- Each sector's PDFs are copied from Drive to local disk before ingesting (reading from Drive while indexing is slow).

**Code:** on your PC run `python -m scripts.common.colab_bundle` and upload `output/colab/capstone_code.zip` to `MyDrive/capstone_rag/`. Cell 2 unpacks it (and again only when you upload a new build). Without a zip it asks you to upload one, or clones `REPO`.

**Cells:** 1-5 ingest, 6-6b check (status and coverage of the 1,004-filing plan), 7 retrieval checks, 9-12 serve the index through FastAPI + Streamlit. The full structure is in [docs/annual_report_rag_structure.md](../docs/annual_report_rag_structure.md).

**Reconnecting:** after a disconnect, run all again. Cell 3 restores the index from Drive, and finished filings are skipped.

**Smoke test** (`SMOKE_TEST = True`, the default):
- 2 PDFs per sector (26 for all 13), 1 filing per sector, 30 pages, 3 VLM pages: the plan's pilot.
- It writes to separate `_smoke` indexes, and embedded Qdrant for `colpali_qdrant`.
- Check the output, then set it to `False` for the real run.

In [ ]:
# @title 1. Settings
PIPELINE = "colpali_qdrant"  # @param ["colpali_qdrant", "vlm_chroma"]
CORPUS = "full"  # @param ["full", "min500"]
# full = all 1,004 planned filings; min500 = 509 (Track B minimum: 41 per sector + every eval-set company)
SECTORS = ""  # @param {type:"string"}
# "" = all 13 sectors (the plan); or e.g. "banking,insurance"
MAX_PER_SECTOR = 0  # @param {type:"integer"}
# 0 = every PDF of the CORPUS preset (1,004 or 509); 20 = a first pass (largest companies first)
COLPALI_PAGES = "tables_charts_images"  # @param ["tables_charts_images", "tables_charts", "visual", "all"]
# ColPali only on table, chart and image pages; all text goes to annual_reports_text either way
VLM_PAGES = "visual"  # @param ["visual", "tables_charts_images", "tables_charts", "all"]
# visual = tables, charts, images + unreadable text pages (without them those pages are missing from Chroma)
OCR_MAX_PAGES_PER_FILING = 0  # @param {type:"integer"}
# colpali_qdrant: OCR every unreadable page so all the text is in annual_reports_text; 20 caps the OCR time
SNAPSHOT_EVERY = 0  # @param {type:"integer"}
# batches of 10 filings between Drive snapshots; 0 = once per sector (each snapshot copies the whole collection)
VLM_MODEL = "Qwen/Qwen3-VL-4B-Instruct"  # @param ["Qwen/Qwen3-VL-4B-Instruct", "Qwen/Qwen3-VL-2B-Instruct", "Qwen/Qwen3-VL-8B-Instruct"]
SMOKE_TEST = True  # @param {type:"boolean"}
SEC_USER_AGENT = "capstone-research your.name@example.com"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/capstone_rag"
REPO = "https://github.com/rama159065/Capstone.git"   # used only when there is no code zip
BRANCH = "finance_report_qa_agent"
CODE_ZIP = "/content/drive/MyDrive/capstone_rag/capstone_code.zip"   # python -m scripts.common.colab_bundle, then upload to Drive

In [ ]:
# @title 2. Drive, code, dependencies
import os
from google.colab import drive
drive.mount("/content/drive")
import json, subprocess, sys
from pathlib import Path


def fetch_code(root, repo_url, branch):
    """Unpack the code zip (Drive, else one you upload) into root; without a zip, clone the repo."""
    import zipfile
    from google.colab import drive
    drive.mount("/content/drive")
    uploaded = Path("/content") / Path(CODE_ZIP).name
    source = next((p for p in (Path(CODE_ZIP), uploaded) if p.exists()), None)
    if source is None and not root.exists():
        from google.colab import files
        print(f"{CODE_ZIP} not found: upload capstone_code.zip (cancel to clone {repo_url} instead)")
        names = list(files.upload())
        source = Path("/content") / names[0] if names else None
    if source is not None:
        with zipfile.ZipFile(source) as archive:
            info = archive.read("BUNDLE_INFO.json")
            stamp = root / "BUNDLE_INFO.json"
            if not stamp.exists() or stamp.read_bytes() != info:      # a new build: unpack over the old one
                archive.extractall(root)
        print("code:", source, json.loads(info))
    elif not root.exists():
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")      # needed while the repo is private
        except Exception:
            token = None
        url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch, url, str(root)], check=True)


fetch_code(Path("/content/Capstone"), REPO, BRANCH)
sys.path.insert(0, "/content/Capstone")
%cd /content/Capstone
!pip -q install "qdrant-client>=1.12,<2" "pymupdf>=1.24,<2" "transformers>=4.57,<5" "sentence-transformers>=3" "rapidocr>=3,<4" onnxruntime
if PIPELINE == "vlm_chroma":
    !pip -q install "chromadb>=1.1" "bitsandbytes>=0.48" accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU: the text stage works, ColPali/VLM will be very slow"

In [ ]:
# @title 3. Settings object, Qdrant server, restore from Drive
import sys
sys.path.insert(0, "/content/Capstone")
from scripts.rag import colab_pipeline as cp
os.environ["SEC_USER_AGENT"] = SEC_USER_AGENT
s = cp.Settings(
    pipeline=PIPELINE, corpus_preset=CORPUS, drive_root=DRIVE_ROOT, sectors=SECTORS, user_agent=SEC_USER_AGENT,
    max_per_sector=2 if SMOKE_TEST else MAX_PER_SECTOR,
    visual_pages=COLPALI_PAGES, vlm_pages=VLM_PAGES, vlm_model=VLM_MODEL, snapshot_every=SNAPSHOT_EVERY,
    ocr_max_pages_per_filing=OCR_MAX_PAGES_PER_FILING,
    run_name="_smoke" if SMOKE_TEST else "",
    qdrant_url=None if SMOKE_TEST else "http://127.0.0.1:6333",      # one server per VM: keep smoke runs off it
    limit_filings=1 if SMOKE_TEST else None,
    max_pages_per_pdf=30 if SMOKE_TEST else None,
    vlm_max_pages_per_pdf=3 if SMOKE_TEST else 0)
cp.prepare(s)
cp.restore(s)

In [ ]:
# @title 4. Download the chosen sectors to Drive (resumable; skips PDFs already there)
cp.download(s)
!du -sh {DRIVE_ROOT}/annual_reports/*/ 2>/dev/null | sort -k2

In [ ]:
# @title 5. Ingest, one sector at a time (rerun after a disconnect: cells 2-3, then this one)
# Each sector's PDFs are copied from Drive to local disk first; a snapshot goes to Drive after each sector.
results = cp.ingest(s)
results

In [ ]:
# @title 6. Status: points per sector, failures, what is in RAM and on disk
report = cp.status(s)

In [ ]:
# @title 6b. Coverage of the plan: every planned filing by sector (planned -> downloaded -> ingested)
cov = cp.coverage(s)
# Full plan done when total ingested == 1,004. "failed" lists filings to retry (rerun cell 5); errors are in cell 6.

In [ ]:
# @title 7. Retrieval checks
cp.query(s, "net interest income", sector="banking")
cp.query(s, "net oil-equivalent production", sector="oil_and_gas")
if PIPELINE == "colpali_qdrant":
    cp.query(s, "revenue by segment bar chart", sector="banking", visual=True)   # pooled prefetch -> full MaxSim rerank

## Serve the index: FastAPI + Streamlit on this runtime

```
browser -> Colab port proxy (only you) -> Streamlit :8501 -> FastAPI :8000 -> this pipeline's index
```

`cp.serve(s)` starts both with the environment from `cp.serving_env(s)`:
- `colpali_qdrant`: `RAG_BACKEND=financial_qdrant`, `RAG_INGEST_DIR=/content/rag/colpali_qdrant`, the Qdrant server (needs `SMOKE_TEST = False`, since smoke runs use embedded Qdrant);
- `vlm_chroma`: `RAG_BACKEND=financial_vlm`, `RAG_VLM_DIR=/content/rag/vlm_chroma`.

Requests can filter by `ticker`, `year` and `sector` (e.g. `"banking"`). Add `LLM_API_BASE`, `LLM_API_KEY`, `LLM_MODEL` as Colab secrets for the provided model; without them only `mock` answers.

In [ ]:
# @title 9. Serving dependencies (from pyproject.toml; keeps Colab's CUDA torch) and LLM secrets
import re, subprocess, tomllib
deps = tomllib.load(open("/content/Capstone/pyproject.toml", "rb"))["project"]["dependencies"]
deps = [d for d in deps if re.split(r"[<>=!~ \[]", d)[0] not in {"torch", "torchvision", "jupyterlab"}]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *deps])
from google.colab import userdata
for key in ("LLM_API_BASE", "LLM_API_KEY", "LLM_MODEL"):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print(f"{key} is not a Colab secret: answers use llm_provider=mock")
# os.environ["RAG_FIN_VISUAL"] = "1"     # colpali_qdrant: add ColPali visual hits to answers (loads the model)

In [ ]:
# @title 10. Start the API and the chat UI
api, ui = cp.serve(s)
import time; time.sleep(8)
from google.colab.output import eval_js
print("Open the chat UI:", eval_js("google.colab.kernel.proxyPort(8501)"))

In [ ]:
# @title 11. Check the API without the UI: sector-filtered retrieval and one answer
import json, urllib.request
def post(path, body):
    request = urllib.request.Request(f"http://127.0.0.1:8000{path}", data=json.dumps(body).encode(),
                                     headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(request, timeout=300))
hits = post("/retrieve/financial_report_qa", {"query": "net interest income", "sector": "banking", "top_k": 3})
print(hits["count"], [e.get("chunk_id") for e in hits.get("evidence_preview", hits.get("evidence", []))][:3])
answer = post("/chat/financial_report_qa", {"query": "What was JPMorgan's net interest income?", "ticker": "JPM",
                                            "year": 2024, "llm_provider": "mock"})
print(answer["final_status"], "|", answer.get("answer", "")[:300])

In [ ]:
# @title 12. Stop serving (the index and Qdrant keep running)
for process in (ui, api):
    process.terminate()

## Next

- **Real run:** set `SMOKE_TEST = False` and run all. Rough cost per sector of about 110 PDFs (**estimates**; check `status` after the first sector):
  - text stage: about 2 h on CPU;
  - ColPali: about 1–3.5 GPU-hours on a T4;
  - VLM: about 10–35 GPU-hours for table and chart pages alone; `visual` adds images and unreadable pages.
  - OCR of unreadable pages (`colpali_qdrant`): not measured yet; `status` shows the seconds per filing.
- **The full plan** (`SECTORS = ""`, `MAX_PER_SECTOR = 0`): 1,004 PDFs, 11.3 GB on Drive. The full ColPali vectors take about 264 KB a page (1030 × 128 float16): about 39 GB if every page went to ColPali, and roughly 17–29 GB for table, chart and image pages (44–75% of pages; `status` after the pilot gives the real count). Either way the Drive snapshots need more than the free 15 GB (e.g. the 100 GB plan). The runtime's local disk holds the PDFs, the live index and one snapshot in progress.
- **VLM at full scale:** one page at a time (`vlm="transformers"`) doesn't fit the budget for ~89k pages. Serve Qwen2.5-VL with vLLM and set `s.vlm = "openai"` with `RAG_VISION_API_BASE` / `RAG_VISION_MODEL`.
- **Fewer sectors:** list them in `SECTORS` (all 13: banking, insurance, financial_services, oil_and_gas, energy_and_utilities,
  healthcare, technology_and_it, retail, telecom_and_media, industrials, consumer_goods, materials_and_chemicals,
  real_estate).
- **Serving on your PC** after copying the index down: the `local` profile reads `indexes/financial_qdrant` (ColPali) or `indexes/financial_vlm` (VLM); see `docs/annual_report_rag_structure.md`.